# Talent Intelligence & Skills Gap Analysis
## 02 — Skill Gap & Role Readiness Engine

**Scenario:** NorthStar Digital Services is fictional. All workforce and talent data are synthetic.

### Objectives
1. Compare each employee's current skills with target-role requirements.
2. Calculate skill-level gaps.
3. Build an interpretable **Role Readiness Score**.
4. Rank internal candidates for each target role.
5. Identify the most important upskilling gaps by role and across the workforce.

### Scoring principle
The readiness model uses **skills only**. Performance rating, age, location, potential category, and other personal attributes are **not used** in the score.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)


### 1. Connect to the GitHub repository in Google Colab
Run this cell if the repository is not already cloned in the current Colab session.


In [ ]:
REPO_URL = 'https://github.com/mazaheriasad/talent-intelligence-skills-gap-analysis.git'
REPO_NAME = 'talent-intelligence-skills-gap-analysis'

if not Path('/content/' + REPO_NAME).exists():
    !git clone {REPO_URL}

%cd /content/{REPO_NAME}


### 2. Load project data


In [ ]:
DATA_DIR = Path('data/raw')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

employee_master = pd.read_csv(DATA_DIR / 'employee_master.csv')
employee_skills = pd.read_csv(DATA_DIR / 'employee_skills.csv')
target_roles = pd.read_csv(DATA_DIR / 'target_roles.csv')
role_skills = pd.read_csv(DATA_DIR / 'role_skills.csv')

print('Employees:', len(employee_master))
print('Employee skill records:', len(employee_skills))
print('Target roles:', len(target_roles))
print('Role skill requirements:', len(role_skills))


### 3. Build the employee × role × required-skill matrix
For every employee and every target role, create one row per required skill. Missing employee skills are treated as proficiency `0`.


In [ ]:
employees = employee_master[['EmployeeID']].drop_duplicates()
roles = target_roles[['RoleID', 'RoleName']].drop_duplicates()

employee_role = employees.merge(roles, how='cross')

comparison = employee_role.merge(
    role_skills[['RoleID','Skill','RequiredProficiency','ImportanceWeight','Mandatory']],
    on='RoleID',
    how='left'
)

comparison = comparison.merge(
    employee_skills[['EmployeeID','Skill','ProficiencyLevel','MonthsSinceLastUse']],
    on=['EmployeeID','Skill'],
    how='left'
)

comparison['ProficiencyLevel'] = comparison['ProficiencyLevel'].fillna(0)
comparison['MonthsSinceLastUse'] = comparison['MonthsSinceLastUse'].fillna(999)

comparison.head()


### 4. Calculate skill attainment and skill gap

`SkillAttainment` is capped at 1.00. Surplus proficiency therefore does not inflate readiness.

`SkillGap = max(RequiredProficiency - CurrentProficiency, 0)`


In [ ]:
comparison['SkillAttainment'] = np.minimum(
    comparison['ProficiencyLevel'] / comparison['RequiredProficiency'],
    1.0
)

comparison['SkillGap'] = np.maximum(
    comparison['RequiredProficiency'] - comparison['ProficiencyLevel'],
    0
)

comparison['WeightedAttainment'] = (
    comparison['SkillAttainment'] * comparison['ImportanceWeight']
)

comparison['MeetsRequirement'] = (
    comparison['ProficiencyLevel'] >= comparison['RequiredProficiency']
)

comparison[['EmployeeID','RoleName','Skill','ProficiencyLevel','RequiredProficiency','SkillGap','SkillAttainment']].head(15)


### 5. Role Readiness Score

The score has two transparent components:

- **85% Weighted Skill Readiness:** importance-weighted attainment across all required skills.
- **15% Mandatory Skill Coverage:** percentage of mandatory skills meeting the target proficiency.

This produces a score from 0 to 100.


In [ ]:
def role_readiness(group):
    weighted_readiness = (
        group['WeightedAttainment'].sum() / group['ImportanceWeight'].sum()
    ) * 100

    mandatory = group[group['Mandatory'].eq('Yes')]
    mandatory_coverage = (
        mandatory['MeetsRequirement'].mean() * 100
        if len(mandatory) > 0 else 100
    )

    total_gap = group['SkillGap'].sum()
    critical_gap_count = (
        (group['SkillGap'].gt(0) & group['ImportanceWeight'].eq(3)).sum()
    )
    missing_skill_count = (group['ProficiencyLevel'].eq(0)).sum()

    final_score = 0.85 * weighted_readiness + 0.15 * mandatory_coverage

    return pd.Series({
        'WeightedSkillReadiness': weighted_readiness,
        'MandatorySkillCoverage': mandatory_coverage,
        'RoleReadinessScore': final_score,
        'TotalSkillGap': total_gap,
        'CriticalGapCount': critical_gap_count,
        'MissingSkillCount': missing_skill_count,
    })

readiness = (
    comparison.groupby(['EmployeeID','RoleID','RoleName'])
    .apply(role_readiness)
    .reset_index()
)

readiness.head()


### 6. Add employee context — but do not use it in the score


In [ ]:
context_cols = [
    'EmployeeID','Department','JobLevel','Location','TenureYears',
    'PerformanceRating','PotentialCategory','MobilityInterest'
]

readiness = readiness.merge(
    employee_master[context_cols],
    on='EmployeeID',
    how='left'
)

def readiness_band(score):
    if score >= 85:
        return 'Ready Now'
    elif score >= 70:
        return 'Near Ready'
    elif score >= 55:
        return 'Developing'
    return 'Significant Gap'

readiness['ReadinessBand'] = readiness['RoleReadinessScore'].apply(readiness_band)
readiness['RoleReadinessScore'] = readiness['RoleReadinessScore'].round(1)
readiness['WeightedSkillReadiness'] = readiness['WeightedSkillReadiness'].round(1)
readiness['MandatorySkillCoverage'] = readiness['MandatorySkillCoverage'].round(1)

readiness.head()


### 7. Top internal candidates by target role


In [ ]:
top_candidates = (
    readiness.sort_values(
        ['RoleID','RoleReadinessScore','MandatorySkillCoverage','CriticalGapCount'],
        ascending=[True,False,False,True]
    )
    .groupby('RoleID', as_index=False)
    .head(10)
    .copy()
)

top_candidates[
    ['RoleName','EmployeeID','Department','JobLevel','RoleReadinessScore',
     'ReadinessBand','MandatorySkillCoverage','CriticalGapCount','MobilityInterest']
].head(20)


### 8. Readiness distribution by role


In [ ]:
role_summary = (
    readiness.groupby(['RoleID','RoleName'])
    .agg(
        AvgReadiness=('RoleReadinessScore','mean'),
        MedianReadiness=('RoleReadinessScore','median'),
        ReadyNow=('ReadinessBand', lambda s: (s == 'Ready Now').sum()),
        NearReady=('ReadinessBand', lambda s: (s == 'Near Ready').sum()),
        Developing=('ReadinessBand', lambda s: (s == 'Developing').sum()),
        SignificantGap=('ReadinessBand', lambda s: (s == 'Significant Gap').sum())
    )
    .reset_index()
    .sort_values('AvgReadiness', ascending=False)
)

role_summary[['RoleName','AvgReadiness','ReadyNow','NearReady','Developing','SignificantGap']].round(1)


In [ ]:
plot_data = role_summary.sort_values('AvgReadiness')
plt.figure(figsize=(10,6))
plt.barh(plot_data['RoleName'], plot_data['AvgReadiness'])
plt.xlabel('Average Role Readiness Score')
plt.ylabel('Target role')
plt.title('Average Workforce Readiness by Target Role')
plt.xlim(0,100)
plt.tight_layout()
plt.show()


### 9. Organization-wide skill-gap priorities


In [ ]:
candidate_keys = top_candidates[['EmployeeID','RoleID']].drop_duplicates()
candidate_gaps = comparison.merge(candidate_keys, on=['EmployeeID','RoleID'], how='inner')

skill_gap_priority = (
    candidate_gaps[candidate_gaps['SkillGap'] > 0]
    .groupby('Skill')
    .agg(
        CandidateGapOccurrences=('EmployeeID','count'),
        AvgGap=('SkillGap','mean'),
        MaxImportance=('ImportanceWeight','max'),
        RolesAffected=('RoleID','nunique')
    )
    .reset_index()
)

skill_gap_priority['PriorityIndex'] = (
    skill_gap_priority['CandidateGapOccurrences'] *
    skill_gap_priority['AvgGap'] *
    skill_gap_priority['MaxImportance']
)

skill_gap_priority = skill_gap_priority.sort_values('PriorityIndex', ascending=False)
skill_gap_priority.head(15).round(2)


### 10. Skill gaps by target role


In [ ]:
role_skill_gaps = (
    candidate_gaps[candidate_gaps['SkillGap'] > 0]
    .groupby(['RoleID','RoleName','Skill'])
    .agg(
        CandidateGapOccurrences=('EmployeeID','count'),
        AvgGap=('SkillGap','mean'),
        RequiredProficiency=('RequiredProficiency','max'),
        ImportanceWeight=('ImportanceWeight','max')
    )
    .reset_index()
)

role_skill_gaps['PriorityIndex'] = (
    role_skill_gaps['CandidateGapOccurrences'] *
    role_skill_gaps['AvgGap'] *
    role_skill_gaps['ImportanceWeight']
)

role_skill_gaps = role_skill_gaps.sort_values(
    ['RoleID','PriorityIndex'], ascending=[True,False]
)

role_skill_gaps.head(30).round(2)


### 11. Examine one candidate in detail


In [ ]:
example = top_candidates.sort_values('RoleReadinessScore', ascending=False).iloc[0]
example_employee = example['EmployeeID']
example_role = example['RoleID']

print('Employee:', example_employee)
print('Target role:', example['RoleName'])
print('Readiness:', example['RoleReadinessScore'])

candidate_detail = comparison[
    (comparison['EmployeeID'] == example_employee) &
    (comparison['RoleID'] == example_role)
][[
    'Skill','ProficiencyLevel','RequiredProficiency','ImportanceWeight',
    'Mandatory','SkillGap','MeetsRequirement'
]].sort_values(['SkillGap','ImportanceWeight'], ascending=[False,False])

candidate_detail


### 12. Save analytical outputs


In [ ]:
readiness.to_csv(OUTPUT_DIR / 'employee_role_readiness.csv', index=False)
top_candidates.to_csv(OUTPUT_DIR / 'top_internal_candidates.csv', index=False)
comparison.to_csv(OUTPUT_DIR / 'employee_role_skill_comparison.csv', index=False)
role_summary.to_csv(OUTPUT_DIR / 'role_readiness_summary.csv', index=False)
skill_gap_priority.to_csv(OUTPUT_DIR / 'skill_gap_priority.csv', index=False)
role_skill_gaps.to_csv(OUTPUT_DIR / 'role_skill_gaps.csv', index=False)

print('Saved:')
for f in [
    'employee_role_readiness.csv',
    'top_internal_candidates.csv',
    'employee_role_skill_comparison.csv',
    'role_readiness_summary.csv',
    'skill_gap_priority.csv',
    'role_skill_gaps.csv'
]:
    print('-', OUTPUT_DIR / f)


## Interpretation note
This model is a **portfolio analytics prototype**, not an automated employment decision system. A real implementation would require validated skills data, employee consent and transparency, governance, bias testing, human review, and organizational calibration of the scoring logic.

## Next phase
Notebook 03 will use NLP to extract skills from free-text job descriptions and compare the extracted skills with the manually structured role requirements.
